In [1]:
!pip install google-generativeai faiss-cpu open_clip_torch==2.23.0 lightning matplotlib -q
!pip install qwen_vl_utils
!pip install transformers --upgrade

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.0/846.0 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 52.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 106.7 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.1
    Uninstalling transformers-4.57.1:
      Successfully uninstalled transformers-4.57.1


In [2]:
import os
import json
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import faiss
import lightning as L
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import ModelCheckpoint
from transformers import AutoTokenizer

In [3]:
from torch.utils.data import Dataset

class ENTREPJsonDataset(Dataset):
    def __init__(self, metadata_file, image_dir, transform=None):
        self.image_dir = image_dir
        self.transform = transform
        self.data = self._load_jsonl(metadata_file)

    def _load_jsonl(self, filepath):
        data = []
        missing = 0

        with open(filepath, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    entry = json.loads(line)
                except:
                    continue

                img_name = entry.get("image")
                text = entry.get("text")

                if not img_name or not text:
                    continue

                full_path = os.path.join(self.image_dir, img_name)

                if os.path.exists(full_path):
                    data.append({
                        "image_name": img_name,
                        "image_path": full_path,
                        "text": str(text)
                    })
                else:
                    missing += 1

        print("===== DATASET INFO =====")
        print(f"Loaded entries     : {len(data)}")
        print(f"Missing image files: {missing}")
        print("========================")

        return data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        try:
            image = Image.open(item["image_path"]).convert("RGB")
        except:
            print(f"[WARNING] Corrupt image: {item['image_path']}")
            image = Image.new("RGB", (224, 224))

        if self.transform:
            image = self.transform(image)

        return {
            "image": image,
            "text": item["text"],
            "image_path": item["image_path"],
            "image_name": item["image_name"]
        }


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [4]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = train_transform

TRAIN_JSONL = "/kaggle/input/entrep/entrep_processed/train/metadata_train.jsonl"
TEST_JSONL = "/kaggle/input/entrep/entrep_processed/test/metadata_test.jsonl"
VAL_JSONL = "/kaggle/input/entrep/entrep_processed/val/metadata_val.jsonl"

IMG_ROOT_TRAIN = "/kaggle/input/entrep/entrep_processed/train/images"
IMG_ROOT_TEST = "/kaggle/input/entrep/entrep_processed/test/images"
IMG_ROOT_VAL = "/kaggle/input/entrep/entrep_processed/val/images"

train_dataset = ENTREPJsonDataset(TRAIN_JSONL, IMG_ROOT_TRAIN, transform=train_transform)
test_dataset  = ENTREPJsonDataset(TEST_JSONL, IMG_ROOT_TEST, transform=test_transform)
val_dataset   = ENTREPJsonDataset(VAL_JSONL, IMG_ROOT_VAL, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

===== DATASET INFO =====
Loaded entries     : 452
Missing image files: 0
===== DATASET INFO =====
Loaded entries     : 58
Missing image files: 0
===== DATASET INFO =====
Loaded entries     : 56
Missing image files: 0


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import lightning as L


# ============================== LOSS ==============================
class ContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, img_emb, txt_emb):
        # cosine similarity
        logits = img_emb @ txt_emb.T / self.temperature

        batch = img_emb.size(0)
        labels = torch.arange(batch, device=img_emb.device)

        loss_i2t = F.cross_entropy(logits, labels)
        loss_t2i = F.cross_entropy(logits.T, labels)

        return (loss_i2t + loss_t2i) / 2


# ============================== IMAGE ENCODER ==============================
class ImageEncoder(nn.Module):
    def __init__(self, out_dim=64, unfreeze_blocks=2):
        super().__init__()

        self.encoder = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14")

        # freeze all layers
        for p in self.encoder.parameters():
            p.requires_grad = False

        # unfreeze the last N transformer blocks
        for block in self.encoder.blocks[-unfreeze_blocks:]:
            for p in block.parameters():
                p.requires_grad = True

        self.head = nn.Sequential(
            nn.Linear(self.encoder.embed_dim, 256),
            nn.GELU(),
            nn.Linear(256, out_dim)
        )

    def forward(self, x):
        feats = self.encoder.forward_features(x)["x_norm_clstoken"]
        return self.head(feats)


# ============================== TEXT ENCODER ==============================
class TextEncoder(nn.Module):
    def __init__(self, out_dim=64, model_name="sentence-transformers/all-MiniLM-L6-v2"):
        super().__init__()

        self.transformer = AutoModel.from_pretrained(model_name)

        # freeze all
        for p in self.transformer.parameters():
            p.requires_grad = False

        # unfreeze last 2 layers
        for layer in self.transformer.encoder.layer[-2:]:
            for p in layer.parameters():
                p.requires_grad = True

        self.head = nn.Sequential(
            nn.Linear(self.transformer.config.hidden_size, 256),
            nn.GELU(),
            nn.Linear(256, out_dim)
        )

    def forward(self, ids, mask):
        x = self.transformer(input_ids=ids, attention_mask=mask).last_hidden_state[:, 0]
        return self.head(x)


# ============================== NANO CLIP MODEL ==============================
class NanoCLIP(L.LightningModule):
    def __init__(self, embed_dim=64, lr=1e-4, warmup_steps=500):
        super().__init__()
        self.save_hyperparameters()

        self.img_encoder = ImageEncoder(embed_dim)
        self.txt_encoder = TextEncoder(embed_dim)
        self.loss_fn = ContrastiveLoss()

        self.tokenizer = AutoTokenizer.from_pretrained(
            "sentence-transformers/all-MiniLM-L6-v2"
        )

        self.warmup_steps = warmup_steps

    # ----------------------------------------------------
    def forward(self, img, ids, mask):
        img_emb = F.normalize(self.img_encoder(img), p=2, dim=-1)
        txt_emb = F.normalize(self.txt_encoder(ids, mask), p=2, dim=-1)
        return img_emb, txt_emb

    # ================= TRAIN =================
    def training_step(self, batch, batch_idx):
        img = batch["image"]
        txt = batch["text"]

        tok = self.tokenizer(
            list(txt),
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        ids = tok["input_ids"].to(self.device)
        mask = tok["attention_mask"].to(self.device)
        img = img.to(self.device)

        img_emb, txt_emb = self(img, ids, mask)
        loss = self.loss_fn(img_emb, txt_emb)

        self.log("train_loss", loss, prog_bar=True, on_epoch=True, on_step=False)
        return loss

    # ================= VALIDATION =================
    def validation_step(self, batch, batch_idx):
        img = batch["image"]
        txt = batch["text"]

        tok = self.tokenizer(
            list(txt),
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        ids = tok["input_ids"].to(self.device)
        mask = tok["attention_mask"].to(self.device)
        img = img.to(self.device)

        img_emb, txt_emb = self(img, ids, mask)
        loss = self.loss_fn(img_emb, txt_emb)

        # BẮT BUỘC: log val_loss để checkpoint hoạt động
        self.log("val_loss", loss, prog_bar=True, on_epoch=True, on_step=False)
        return loss

    # ================= OPTIMIZER =================
    def configure_optimizers(self):
        opt = torch.optim.AdamW(
            self.parameters(), lr=self.hparams.lr, weight_decay=1e-3
        )

        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            opt, T_max=20, eta_min=1e-6
        )

        return {
            "optimizer": opt,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "epoch",
                "frequency": 1,
            }
        }


In [6]:
checkpoint = ModelCheckpoint(
    save_top_k=1,
    monitor="val_loss",
    mode="min",
    filename="nanoclip-{epoch:02d}-{val_loss:.4f}"
)

trainer = Trainer(
    max_epochs=10,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    callbacks=[checkpoint],
    log_every_n_steps=5,               # logging thường xuyên hơn
    check_val_every_n_epoch=1,         # BẮT BUỘC: chạy val mỗi epoch
    enable_checkpointing=True          # đảm bảo Lightning bật checkpoint
)

model = NanoCLIP(embed_dim=64, lr=1e-4)

trainer.fit(model, train_loader, val_loader)


Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores


Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 213MB/s]


...

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/data.py:79: Trying to infer the `batch_size` 
from an ambiguous collection. The batch size we found is 4. To avoid any miscalculations, use `self.log(..., 
batch_size=batch_size)`.

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/data.py:79: Trying to infer the `batch_size` 
from an ambiguous collection. The batch size we found is 8. To avoid any miscalculations, use `self.log(..., 
batch_size=batch_size)`.

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

In [7]:
best_path = checkpoint.best_model_path
print("Best checkpoint:", best_path)

best_model = NanoCLIP.load_from_checkpoint(best_path, strict=False)
best_model = best_model.to("cuda")
best_model.eval()
# 🔥 BẮT BUỘC: move encoder weights lên CUDA
best_model.img_encoder.to("cuda")
best_model.txt_encoder.to("cuda")
print("Encoders moved to CUDA!")


image_embeddings = []
image_paths = []

# đảm bảo test_dataset trả về {"image": tensor, "image_path": string}
for item in test_dataset:
    img = item["image"].unsqueeze(0).to("cuda")

    with torch.no_grad():
        # dùng forward encoder giống training
        img_emb = best_model.img_encoder(img)
        img_emb = F.normalize(img_emb, p=2, dim=-1)

    image_embeddings.append(img_emb.cpu().numpy()[0])
    image_paths.append(item["image_path"])

# convert to FAISS format
image_embeddings = np.vstack(image_embeddings).astype("float32")

faiss_index = faiss.IndexFlatL2(image_embeddings.shape[1])
faiss_index.add(image_embeddings)

print("FAISS index built:", faiss_index.ntotal)


Best checkpoint: /kaggle/working/lightning_logs/version_0/checkpoints/nanoclip-epoch=03-val_loss=1.3758.ckpt


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


Encoders moved to CUDA!
FAISS index built: 58


In [8]:
def retrieve(query, k=5):
    # Dùng đúng best_model đã load từ checkpoint
    tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
    tok = tokenizer(query, return_tensors="pt", padding=True, truncation=True, max_length=128)

    ids = tok["input_ids"].to("cuda")
    mask = tok["attention_mask"].to("cuda")

    with torch.no_grad():
        # Dùng best_model thay vì model
        txt_emb = best_model.txt_encoder(ids, mask)
        txt_emb = F.normalize(txt_emb, p=2, dim=-1).cpu().numpy().astype("float32")

    # FAISS search
    D, I = faiss_index.search(txt_emb, k)

    idxs = I[0]

    print("🔍 Query:", query)
    print("===================")

    for i in idxs:
        print("→", image_paths[i])

    return idxs



retrieve("edema and erythema of the arytenoid cartilage")

🔍 Query: edema and erythema of the arytenoid cartilage
→ /kaggle/input/entrep/entrep_processed/test/images/23114708_230519104404520831_954_Image01.png
→ /kaggle/input/entrep/entrep_processed/test/images/15005172_230913080858365831_954_Image03.png
→ /kaggle/input/entrep/entrep_processed/test/images/16144146_230815100248013831_954_Image06.png
→ /kaggle/input/entrep/entrep_processed/test/images/19010762_231025091014395831_954_Image06.png
→ /kaggle/input/entrep/entrep_processed/test/images/19082461_231011101231274831_954_Image07.png


array([40, 53, 17, 16, 52])

In [9]:
def evaluate_query(gt_list, retrieved_list, k_list=[1,5,10]):
    # Convert paths → filenames
    gt_set = set([os.path.basename(x) for x in gt_list])
    ret_names = [os.path.basename(x) for x in retrieved_list]

    n_retrieved = len(ret_names)
    metrics = {}

    # Recall & Precision
    for k in k_list:
        topk = ret_names[:k]
        hits = sum(1 for img in topk if img in gt_set)

        metrics[f"Recall@{k}"] = hits / len(gt_set)
        metrics[f"Precision@{k}"] = hits / k

    # MRR
    rr = 0.0
    for i, img in enumerate(ret_names):
        if img in gt_set:
            rr = 1.0 / (i + 1)
            break
    metrics["MRR"] = rr

    # NDCG
    import math
    for k in k_list:
        dcg = 0
        for i in range(min(k, n_retrieved)):
            if ret_names[i] in gt_set:
                dcg += 1 / math.log2(i + 2)

        ideal = min(len(gt_set), k)
        idcg = sum(1 / math.log2(i + 2) for i in range(ideal))
        metrics[f"NDCG@{k}"] = dcg / idcg if idcg > 0 else 0.0

    # AP
    hit = 0
    precisions = []
    for i, img in enumerate(ret_names):
        if img in gt_set:
            hit += 1
            precisions.append(hit / (i+1))

    metrics["AP"] = sum(precisions) / len(gt_set) if len(gt_set) > 0 else 0.0

    return metrics


In [10]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
import torch
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading Qwen2-VL-7B-Instruct...")
model_qwen = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-7B-Instruct",
    torch_dtype="auto",
    device_map="auto",
)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-7B-Instruct")
print("Qwen2-VL loaded!")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading Qwen2-VL-7B-Instruct...


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

Qwen2-VL loaded!


In [11]:
import time, random, re

def safe_generate(prompt, image=None, retry=3):

    # Format cho Qwen2-VL theo chat template
    messages = [
        {
            "role": "user",
            "content": []
        }
    ]

    if image is not None:
        messages[0]["content"].append({"type": "image", "image": image})

    messages[0]["content"].append({"type": "text", "text": prompt})

    for _ in range(retry):
        try:
            # Convert message thành model inputs
            text_template = processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )

            image_inputs, video_inputs = process_vision_info(messages)

            inputs = processor(
                text=[text_template],
                images=image_inputs,
                videos=video_inputs,
                return_tensors="pt",
                padding=True
            ).to(device)

            # Generate
            out_ids = model_qwen.generate(
                **inputs, max_new_tokens=20
            )

            # Remove prompt tokens
            gen_ids = out_ids[0][inputs.input_ids.shape[1]:]
            output = processor.decode(
                gen_ids,
                skip_special_tokens=True
            ).strip()

            # Parse float
            try:
                return float(output)
            except:
                nums = re.findall(r"[01](?:\.\d+)?", output)
                if nums:
                    return float(nums[0])

            return 0.0

        except Exception as e:
            print("Retry due to:", e)
            time.sleep(1.3 + random.random())

    return 0.0

In [12]:
TEXT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL)

In [13]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [14]:
def retrieve(query_text, top_k=10):
    tok = tokenizer(query_text, return_tensors="pt").to(device)
    with torch.no_grad():
        txt_emb = best_model.txt_encoder(tok["input_ids"], tok["attention_mask"])
        txt_emb = txt_emb.cpu().numpy().astype("float32")

    D, I = faiss_index.search(txt_emb, top_k)
    clip_scores = 1/(1+D[0])

    results = []
    for rank, idx in enumerate(I[0]):
        img_path = image_paths[idx]
        img = Image.open(img_path).convert("RGB")

        gem_score = safe_generate(
            f"Rate similarity to: {query_text}. Return only a float 0-1.",
            img
        )

        final = 0.6 * clip_scores[rank] + 0.4 * gem_score
        results.append((final, gem_score, clip_scores[rank], img_path))

    results.sort(reverse=True)
    return results


In [15]:
from collections import defaultdict

def build_test_ground_truth(test_dataset):
    GT = defaultdict(list)
    for item in test_dataset:
        text = item["text"].strip()
        img  = item["image_name"]
        GT[text].append(img)
    return GT

GT_TEST = build_test_ground_truth(test_dataset)
print("Total queries:", len(GT_TEST))


Total queries: 41


In [16]:
import numpy as np

# ---------- DCG ----------
def dcg(relevance):
    """
    relevance: list [0/1] theo rank
    """
    return np.sum([
        rel / np.log2(idx + 2)
        for idx, rel in enumerate(relevance)
    ])

# ---------- nDCG@K ----------
def ndcg_at_k(relevance, k):
    rel_k = relevance[:k]
    ideal = sorted(rel_k, reverse=True)

    dcg_val = dcg(rel_k)
    idcg_val = dcg(ideal)

    return dcg_val / (idcg_val + 1e-9)

# ---------- MRR ----------
def mrr(relevance):
    for idx, r in enumerate(relevance):
        if r == 1:
            return 1.0 / (idx + 1)
    return 0.0

# ---------- Recall@K ----------
def recall_at_k(relevance, k):
    return 1.0 if np.sum(relevance[:k]) > 0 else 0.0

# ---------- Precision@K ----------
def precision_at_k(relevance, k):
    return np.sum(relevance[:k]) / k


In [17]:
def average_precision(pred_list, gt_list):
    hits = 0
    sum_precisions = 0.0

    for i, p in enumerate(pred_list):
        if p in gt_list:
            hits += 1
            sum_precisions += hits / (i + 1)

    if hits == 0:
        return 0.0

    return sum_precisions / hits


In [18]:
import os

def evaluate_query(query_text, retrieved_results, ground_truth_images, top_k=10):
    retrieved_paths = [r[3] for r in retrieved_results]

    relevance = [
        1 if os.path.basename(p) in ground_truth_images else 0
        for p in retrieved_paths
    ]

    return {
        "query": query_text,

        "recall@1": recall_at_k(relevance, 1),
        "recall@5": recall_at_k(relevance, 5),
        "recall@10": recall_at_k(relevance, 10),

        "precision@1": precision_at_k(relevance, 1),
        "precision@5": precision_at_k(relevance, 5),
        "precision@10": precision_at_k(relevance, 10),

        "mrr": mrr(relevance),

        "ndcg@1": ndcg_at_k(relevance, 1),
        "ndcg@5": ndcg_at_k(relevance, 5),
        "ndcg@10": ndcg_at_k(relevance, 10),

        # giữ lại để tính mAP
        "retrieved_list": [os.path.basename(p) for p in retrieved_paths],
        "gt_list": list(ground_truth_images),
    }


In [19]:
import pandas as pd
from tqdm import tqdm

eval_rows = []
TOP_K = 10

for q_text, gt_imgs in tqdm(GT_TEST.items()):
    retrieved = retrieve(q_text, top_k=TOP_K)
    metrics = evaluate_query(q_text, retrieved, gt_imgs, top_k=TOP_K)
    eval_rows.append(metrics)

df_eval = pd.DataFrame(eval_rows)

# ---------- AP & mAP ----------
df_eval["AP"] = df_eval.apply(
    lambda row: average_precision(row["retrieved_list"], row["gt_list"]),
    axis=1
)

mAP = df_eval["AP"].mean()
print("mAP:", mAP)

df_eval.to_csv("entrep_test_eval.csv", index=False)


100%|██████████| 41/41 [24:15<00:00, 35.51s/it]

mAP: 0.3755565234223771


In [20]:
print("===== FINAL EVALUATION (WITH RERANKING) =====")

for col in df_eval.columns:
    if df_eval[col].dtype == "object":
        continue
    print(f"{col}: {df_eval[col].mean():.4f}")


===== FINAL EVALUATION (WITH RERANKING) =====
recall@1: 0.1951
recall@5: 0.6585
recall@10: 0.9268
precision@1: 0.1951
precision@5: 0.1463
precision@10: 0.1220
mrr: 0.3999
ndcg@1: 0.1951
ndcg@5: 0.4318
ndcg@10: 0.5153
AP: 0.3756
